### Load in the Model (Llama-3.1-8B-Instruct)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B-Instruct", dtype="float16", device_map="cuda")
print(model.device)
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

cuda:0


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


I'm an artificial intelligence model known as Llama. Llama stands for "Large Language Model Meta AI."<|eot_id|>


### Generating Activations

In [2]:
import pandas as pd

# Load in the Factual Datasets
F0_train, F0_test = pd.read_csv("../datasets/plain_dataset/F0_train.csv")[["statement", "label"]], pd.read_csv("../datasets/plain_dataset/F0_test.csv")[["statement", "label"]]
F1_train, F1_test = pd.read_csv("../datasets/plain_dataset/F1_train.csv")[["statement", "label"]], pd.read_csv("../datasets/plain_dataset/F1_test.csv")[["statement", "label"]]
F2_train, F2_test = pd.read_csv("../datasets/plain_dataset/F2_train.csv"), pd.read_csv("../datasets/plain_dataset/F2_test.csv")
F3_train, F3_test = pd.read_csv("../datasets/plain_dataset/F3_train.csv"), pd.read_csv("../datasets/plain_dataset/F3_test.csv")
F4_train, F4_test = pd.read_csv("../datasets/plain_dataset/F4_train.csv"), pd.read_csv("../datasets/plain_dataset/F4_test.csv")
F5_train, F5_test = pd.read_csv("../datasets/plain_dataset/F5_train.csv"), pd.read_csv("../datasets/plain_dataset/F5_test.csv")

# Load in the Arithmatic Statements
A1_train, A1_test = pd.read_csv("../datasets/plain_dataset/A1_train.csv"), pd.read_csv("../datasets/plain_dataset/A1_test.csv")
A2_train, A2_test = pd.read_csv("../datasets/plain_dataset/A2_train.csv"), pd.read_csv("../datasets/plain_dataset/A2_test.csv")
A3_train, A3_test = pd.read_csv("../datasets/plain_dataset/A3_train.csv"), pd.read_csv("../datasets/plain_dataset/A3_test.csv")


In [3]:
datasets = {
    "F0_train": F0_train, "F0_test": F0_test,
    "F1_train": F1_train, "F1_test": F1_test,
    "F2_train": F2_train, "F2_test": F2_test,
    "F3_train": F3_train, "F3_test": F3_test,
    "F4_train": F4_train, "F4_test": F4_test,
    "F5_train": F5_train, "F5_test": F5_test,
    "A1_train": A1_train, "A1_test": A1_test,
    "A2_train": A2_train, "A2_test": A2_test,
    "A3_train": A3_train, "A3_test": A3_test,
}

for name, df in datasets.items():
    print(f"{name}: {len(df)}")


F0_train: 1194
F0_test: 512
F1_train: 1194
F1_test: 512
F2_train: 1194
F2_test: 512
F3_train: 1398
F3_test: 600
F4_train: 1394
F4_test: 598
F5_train: 1383
F5_test: 593
A1_train: 700
A1_test: 300
A2_train: 700
A2_test: 300
A3_train: 700
A3_test: 300


In [4]:
import torch
from tqdm import tqdm

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

def generate_activations(model, statements, layer, with_chat_template=True, batch_size=16):
    statements = list(statements)
    final_token_activations = []

    for i in range(0, len(statements), batch_size):
        statements_temp = statements[i: i+batch_size]
        if with_chat_template:
            messages = [[{"role": "user", "content": s}] for s in statements_temp]
            inputs = tokenizer.apply_chat_template(
                messages,
                add_generation_prompt=True,
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
                padding=True,
            ).to(model.device)
        else:
            inputs = tokenizer(statements_temp, return_tensors="pt", padding=True).to(model.device)

        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)

        final_token_activation = outputs.hidden_states[layer][:, -1, :]
        final_token_activations.append(final_token_activation.cpu())

    return torch.cat(final_token_activations, dim=0)

In [5]:
for dataset in tqdm([F0_test, F0_train,
                     F1_train, F1_test,
                     F2_train, F2_test,
                     F3_train, F3_test,
                     F4_test, F4_train,
                     F5_train, F5_test, 
                     A1_test, A1_train,
                     A2_test, A2_train, 
                     A3_test, A3_train]):
    activations = generate_activations(model, dataset["statement"], 16, with_chat_template=True, batch_size=16)
    dataset["activations_chat"] = list(activations)
    activations = generate_activations(model, dataset["statement"], 16, with_chat_template=False, batch_size=16)
    dataset["activations"] = list(activations)

100%|██████████| 18/18 [05:24<00:00, 18.05s/it]


In [6]:
F0_test["activations"]

0      [tensor(-0.0793, dtype=torch.float16), tensor(...
1      [tensor(0.0138, dtype=torch.float16), tensor(0...
2      [tensor(-0.0990, dtype=torch.float16), tensor(...
3      [tensor(0.0270, dtype=torch.float16), tensor(0...
4      [tensor(0.0009, dtype=torch.float16), tensor(0...
                             ...                        
507    [tensor(-0.0951, dtype=torch.float16), tensor(...
508    [tensor(0.0264, dtype=torch.float16), tensor(0...
509    [tensor(0.0079, dtype=torch.float16), tensor(0...
510    [tensor(-0.1339, dtype=torch.float16), tensor(...
511    [tensor(-0.0796, dtype=torch.float16), tensor(...
Name: activations, Length: 512, dtype: object

### Training the Model and Extracting AUROC Scores

In [7]:
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

def train_probe_pytorch(activations, labels, device='cuda'):
    (X_train, X_test), (y_train, y_test) = activations, labels

    X_train = torch.stack(list(X_train)).float().numpy()
    X_test = torch.stack(list(X_test)).float().numpy()
    y_train = y_train.to_numpy()
    y_test = y_test.to_numpy()


    # Mean-center using only the training mean
    train_mean = X_train.mean(axis=0)
    X_train = X_train - train_mean
    X_test = X_test - train_mean

    # Convert to tensors
    X_train_t = torch.tensor(X_train, dtype=torch.float32, device=device)
    y_train_t = torch.tensor(y_train, dtype=torch.float32, device=device)
    X_test_t = torch.tensor(X_test, dtype=torch.float32, device=device)

    hidden_dim = X_train.shape[1]

    # THIS is the entire "model": one linear layer, no bias.
    # w(x) = w^T x, no offset term -> passes through the origin.
    probe = nn.Linear(hidden_dim, 1, bias=False).to(device)

    optimizer = torch.optim.Adam(probe.parameters(), lr=1e-3, weight_decay=0.1)
    loss_fn = nn.BCEWithLogitsLoss()  # sigmoid + binary cross-entropy, combined for numerical stability

    for step in range(1000):
        optimizer.zero_grad()
        logits = probe(X_train_t).squeeze(-1)   # w^T x for every example
        loss = loss_fn(logits, y_train_t)
        loss.backward()
        optimizer.step()

    # Evaluate
    probe.eval()
    with torch.no_grad():
        test_logits = probe(X_test_t).squeeze(-1).cpu().numpy()
    auroc = roc_auc_score(y_test, test_logits)

    return probe, train_mean, auroc

In [8]:
train_probe_pytorch((F0_train["activations"], F0_test["activations"]), (F0_train["label"], F0_test["label"]))

(Linear(in_features=4096, out_features=1, bias=False),
 array([-0.04358536,  0.02481364,  0.06437364, ...,  0.08228251,
         0.15625978,  0.12397505], shape=(4096,), dtype=float32),
 0.99945068359375)

In [9]:
train_probe_pytorch((F1_train["activations"], F1_test["activations"]), (F1_train["label"], F1_test["label"]))

(Linear(in_features=4096, out_features=1, bias=False),
 array([ 0.09350186,  0.04340456,  0.15155569, ...,  0.03238351,
         0.0552111 , -0.07839764], shape=(4096,), dtype=float32),
 0.9999542236328125)

In [10]:
train_probe_pytorch((F2_train["activations"], F2_test["activations"]), (F2_train["label"], F2_test["label"]))

(Linear(in_features=4096, out_features=1, bias=False),
 array([ 0.07751152, -0.14075711, -0.05583457, ..., -0.03307418,
        -0.03007445,  0.21405198], shape=(4096,), dtype=float32),
 0.99951171875)

In [11]:
train_probe_pytorch((F3_train["activations"], F3_test["activations"]), (F3_train["label"], F3_test["label"]))

(Linear(in_features=4096, out_features=1, bias=False),
 array([-0.01099828,  0.01098878,  0.05059361, ..., -0.00622188,
        -0.00630282,  0.09386266], shape=(4096,), dtype=float32),
 0.9107666666666666)

In [12]:
train_probe_pytorch((F4_train["activations"], F4_test["activations"]), (F4_train["label"], F4_test["label"]))

(Linear(in_features=4096, out_features=1, bias=False),
 array([-0.07477886,  0.06279498,  0.00620259, ..., -0.00398758,
        -0.15763775,  0.09145942], shape=(4096,), dtype=float32),
 0.7961767765461236)

In [13]:
train_probe_pytorch((F5_train["activations"], F5_test["activations"]), (F5_train["label"], F5_test["label"]))

(Linear(in_features=4096, out_features=1, bias=False),
 array([-0.06708997,  0.00258622, -0.07824166, ...,  0.01494019,
        -0.11020023,  0.11612783], shape=(4096,), dtype=float32),
 0.7351556101556102)

In [14]:
train_probe_pytorch((A1_train["activations"], A1_test["activations"]), (A1_train["label"], A1_test["label"]))

(Linear(in_features=4096, out_features=1, bias=False),
 array([-0.10054926, -0.1698112 ,  0.06591169, ...,  0.0347801 ,
         0.05990821,  0.00083482], shape=(4096,), dtype=float32),
 0.6445777777777777)

In [15]:
train_probe_pytorch((A2_train["activations"], A2_test["activations"]), (A2_train["label"], A2_test["label"]))

(Linear(in_features=4096, out_features=1, bias=False),
 array([-0.11967925, -0.09901559,  0.05275749, ..., -0.03800526,
         0.00885455, -0.03281679], shape=(4096,), dtype=float32),
 0.5796)

In [16]:
train_probe_pytorch((A3_train["activations"], A3_test["activations"]), (A3_train["label"], A3_test["label"]))

(Linear(in_features=4096, out_features=1, bias=False),
 array([-0.15804632, -0.10450189,  0.04143262, ..., -0.03563659,
         0.02630921, -0.02770448], shape=(4096,), dtype=float32),
 0.5740000000000001)

In [17]:
train_probe_pytorch((F0_train["activations_chat"], F0_test["activations_chat"]), (F0_train["label"], F0_test["label"]))

(Linear(in_features=4096, out_features=1, bias=False),
 array([ 0.00439368,  0.01325074, -0.13224043, ..., -0.02771102,
        -0.06419451,  0.01452859], shape=(4096,), dtype=float32),
 0.9997406005859375)

In [18]:
train_probe_pytorch((F1_train["activations_chat"], F1_test["activations_chat"]), (F1_train["label"], F1_test["label"]))

(Linear(in_features=4096, out_features=1, bias=False),
 array([ 0.06795434,  0.00945881, -0.10248821, ..., -0.06401634,
        -0.07949793,  0.03989315], shape=(4096,), dtype=float32),
 0.99993896484375)

In [19]:
train_probe_pytorch((F2_train["activations_chat"], F2_test["activations_chat"]), (F2_train["label"], F2_test["label"]))

(Linear(in_features=4096, out_features=1, bias=False),
 array([ 3.9132573e-02,  1.1755324e-01, -4.1137388e-05, ...,
        -1.3403967e-01, -1.0121833e-01,  6.0078237e-02],
       shape=(4096,), dtype=float32),
 0.9998931884765625)

In [20]:
train_probe_pytorch((F3_train["activations_chat"], F3_test["activations_chat"]), (F3_train["label"], F3_test["label"]))

(Linear(in_features=4096, out_features=1, bias=False),
 array([ 0.09114216,  0.11052811,  0.0215665 , ..., -0.10798901,
        -0.15984033,  0.01066289], shape=(4096,), dtype=float32),
 0.9613444444444446)

In [21]:
train_probe_pytorch((F4_train["activations_chat"], F4_test["activations_chat"]), (F4_train["label"], F4_test["label"]))

(Linear(in_features=4096, out_features=1, bias=False),
 array([ 0.15846415,  0.15725625, -0.00440259, ..., -0.04144637,
        -0.09864468, -0.00487346], shape=(4096,), dtype=float32),
 0.8448115792888222)

In [22]:
train_probe_pytorch((F5_train["activations_chat"], F5_test["activations_chat"]), (F5_train["label"], F5_test["label"]))

(Linear(in_features=4096, out_features=1, bias=False),
 array([ 0.14608411,  0.13761473, -0.03394521, ..., -0.04039997,
        -0.08707612,  0.02404252], shape=(4096,), dtype=float32),
 0.7964896714896714)

In [23]:
train_probe_pytorch((A1_train["activations_chat"], A1_test["activations_chat"]), (A1_train["label"], A1_test["label"]))

(Linear(in_features=4096, out_features=1, bias=False),
 array([ 0.11695445,  0.06046747,  0.05797121, ..., -0.06279703,
        -0.04388838,  0.04010627], shape=(4096,), dtype=float32),
 0.6445777777777778)

In [24]:
train_probe_pytorch((A2_train["activations_chat"], A2_test["activations_chat"]), (A2_train["label"], A2_test["label"]))

(Linear(in_features=4096, out_features=1, bias=False),
 array([ 0.18006985,  0.08285514,  0.07934352, ..., -0.03520111,
        -0.03425208, -0.04565858], shape=(4096,), dtype=float32),
 0.5273777777777777)

In [25]:
train_probe_pytorch((A3_train["activations_chat"], A3_test["activations_chat"]), (A3_train["label"], A3_test["label"]))

(Linear(in_features=4096, out_features=1, bias=False),
 array([ 0.20059204,  0.08768764,  0.06997777, ..., -0.03213523,
        -0.02398991, -0.06627693], shape=(4096,), dtype=float32),
 0.5029777777777777)

We find that chat-template application has negligible effect on simple factual tasks (F0, F1) and simple arithmetic (A1), a small positive effect on complex factual tasks (F4), but substantially degrades performance on the hardest arithmetic task (A3, dropping to near-chance). Given that raw-text prompting produces equal-or-better performance across all tested tasks except F4, and that Poulis's prompt-template examples (Appendix A.3) are shown without chat-formatting, we adopt no-chat-template as our primary methodology, consistent with the likely convention used in the source paper.